# DenseNet121 — Run All จนได้ Final และ External Report

เลือก Python kernel ของ **RTX 5070 Ti environment ตาม locked protocol** แล้วกด **Run All** ได้เลย Notebook จะตรวจชุดข้อมูล 431,672 ภาพ → ฝึก E1 ใหม่และ E2 → เปรียบเทียบ validation → เลือก checkpoint ด้วย Macro F1 (Accuracy เป็นเกณฑ์รอง) → Freeze artifact → ตรวจ inference → สร้างชุด Burapha 72 คลาสที่ตรง label → ประเมิน External และแสดงผลด้านล่าง

**กติกา E5 ของรอบอัตโนมัติ:** ข้าม E5 และบันทึกเหตุผลไว้ เพราะยังไม่มี class-specific intervention ที่ออกแบบจากภาพผิด E1/E2 ไว้ก่อนฝึก หากต้องการอ้างผล E5 จริง ต้องตรวจภาพและกำหนดวิธีนั้นก่อนฝึกใหม่ ใช้ external เฉพาะหลัง Freeze; ไม่ใช้เลือกโมเดล

รัน E1/E2 ใช้ seed 42, เพดาน 50 epochs, early stopping patience 10 และเปลี่ยนเฉพาะ augmentation เวลารันเต็มขึ้นอยู่กับ GPU/early stopping Notebook บันทึก log และผลทั้งหมดใน `results/final/densenet121_e1_e2/`


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src/run_e1_e2.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('เปิด notebook จากภายใน repository')
PROTOCOL = ROOT / 'configs/experiments/densenet121_final_protocol.json'
OUTPUT_DIR = ROOT / 'results/final/densenet121_e1_e2'
CHECK_ONLY = False  # True = ตรวจ dataset/code/environment อย่างเดียว ไม่ฝึก
VERIFY_CONTENT_HASHES = False  # True = ตรวจ SHA256 ของรูปทั้งหมดด้วย (ใช้เวลานานขึ้น)
print('Python:', sys.executable)
print('Protocol:', PROTOCOL)
print('Output:', OUTPUT_DIR)


## รันต่อเนื่อง

Log จะไหลในเซลล์และบันทึกลง `OUTPUT_DIR/console.log`; ดู epoch ที่ `runs/E1_NEW/` และ `runs/E2/` ได้ระหว่างรัน

เปิดซ้ำแล้วใช้ผลรันที่จบสมบูรณ์และเงื่อนไขตรงกันได้ แต่รันที่ขาดช่วง **เริ่ม epoch 1 ใหม่** ไม่ใช่ resume optimizer/scheduler หากโปรแกรมถูก kill จนค้าง `RUNNING.lock` ให้ตรวจว่า process เดิมหยุดจริงก่อนลบไฟล์นั้น


In [ ]:
import subprocess
from datetime import datetime

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-u', '-m', 'src.run_e1_e2',
           '--protocol', str(PROTOCOL), '--output-dir', str(OUTPUT_DIR)]
if CHECK_ONLY:
    command.append('--check-only')
if VERIFY_CONTENT_HASHES:
    command.append('--verify-content-hashes')

with (OUTPUT_DIR / 'console.log').open('a', encoding='utf-8') as log:
    log.write(f'\n=== {datetime.now().isoformat()} ===\n')
    log.flush()
    process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        returncode = process.wait()
        if returncode:
            raise RuntimeError(f'Runner stopped (exit {returncode}); ดู console.log ด้านบน')
    finally:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        process.stdout.close()


In [ ]:
if not CHECK_ONLY:
    import json
    import pandas as pd
    from IPython.display import display, FileLink
    display(pd.read_csv(OUTPUT_DIR / 'comparison.csv')[
        ['experiment', 'val_macro_f1', 'val_accuracy', 'best_epoch', 'epochs_completed', 'training_seconds']])
    print(json.loads((OUTPUT_DIR / 'status.json').read_text(encoding='utf-8')))
    for path in [OUTPUT_DIR / 'REVIEW_NEXT.md', OUTPUT_DIR / 'error_analysis/index.html']:
        print(path)
        display(FileLink(str(path)))


## ผลของ E1/E2 และรายงานภาพผิด

เซลล์ต่อไปใช้ผล validation เพื่อเลือกโมเดล หลังจากนั้นจึง Freeze และเริ่มอ่านชุด External ให้เปิด `error_analysis/index.html` เพื่อดูภาพผิดของแต่ละรันเพิ่มเติม


In [ ]:
if not CHECK_ONLY:
    import hashlib
    import shutil
    from IPython.display import Image, Markdown
    from src.inference import Predictor

    selected = json.loads((OUTPUT_DIR / 'best_e1_e2_candidate.json').read_text(encoding='utf-8'))
    source_checkpoint = Path(selected['checkpoint_path'])
    source_run = source_checkpoint.parent
    source_config = source_run / 'config.json'
    source_metrics = source_run / 'metrics.json'
    source_labels = ROOT / 'data/splits/1BnPkvJJlE7QDZ0sgEDujS23Pr8Cj3IKq_source_v3/label_to_index.json'
    if any(not p.is_file() for p in (source_checkpoint, source_config, source_metrics, source_labels)):
        raise FileNotFoundError('ผลรันที่เลือกขาด checkpoint/config/metrics/labels')

    FINAL_DIR = OUTPUT_DIR / 'final'
    FINAL_DIR.mkdir(parents=True, exist_ok=True)
    for source, name in ((source_checkpoint, 'best_model.pt'),
                         (source_config, 'best_config.json'),
                         (source_labels, 'label_to_index.json'),
                         (source_metrics, 'validation_metrics.json')):
        destination = FINAL_DIR / name
        if destination.exists():
            if hashlib.sha256(source.read_bytes()).hexdigest() != hashlib.sha256(destination.read_bytes()).hexdigest():
                raise ValueError(f'Frozen artifact changed: {destination}')
        else:
            shutil.copy2(source, destination)

    predictor = Predictor(FINAL_DIR / 'best_model.pt', device='auto',
                          config_path=FINAL_DIR / 'best_config.json',
                          labels_path=FINAL_DIR / 'label_to_index.json')
    if predictor.sha256 != selected['checkpoint_sha256'] or len(predictor.labels) != 72:
        raise ValueError('Checkpoint SHA256 or class count differs from selected validation run')
    frozen = {'selected_experiment': selected['experiment'],
              'checkpoint_sha256': predictor.sha256,
              'split_hash': selected['split_hash'],
              'validation_macro_f1': selected['val_macro_f1'],
              'validation_accuracy': selected['val_accuracy'],
              'e5': 'skipped: no prespecified class-specific policy after E1/E2 error review',
              'inference': {'tta': 'none', 'top_k': 3, 'confidence_threshold': 0.0,
                            'image_size': predictor.config.image_size,
                            'architecture': predictor.config.architecture},
              'artifacts_sha256': {name: hashlib.sha256((FINAL_DIR / name).read_bytes()).hexdigest()
                                   for name in ('best_model.pt', 'best_config.json', 'label_to_index.json', 'validation_metrics.json')}}
    freeze_path = FINAL_DIR / 'freeze.json'
    if freeze_path.exists() and json.loads(freeze_path.read_text(encoding='utf-8')) != frozen:
        raise ValueError('Frozen model metadata changed; do not overwrite an existing final export')
    freeze_path.write_text(json.dumps(frozen, ensure_ascii=False, indent=2), encoding='utf-8')
    example_path = ROOT / 'notebooks/assets/example_ก.png'
    smoke = predictor.predict([example_path], top_k=3, tta='none')
    if len(smoke) != 1 or smoke[0]['status'] != 'OK':
        raise RuntimeError(f'Final inference smoke test failed: {smoke}')
    (FINAL_DIR / 'inference_smoke.json').write_text(json.dumps(smoke, ensure_ascii=False, indent=2), encoding='utf-8')
    display(Markdown(f"**Final selected:** {selected['experiment']} · validation Macro F1 {selected['val_macro_f1']:.6f} · SHA256 `{predictor.sha256}`"))
    display(Markdown('### ภาพตัวอย่างและผลทำนาย'))
    display(Image(filename=str(example_path), width=192))
    display(Markdown(f"**ทำนาย:** {smoke[0]['label']} · **ความมั่นใจ:** {smoke[0]['confidence']:.2%} · **สถานะ:** {smoke[0]['status']}"))
    display(pd.DataFrame(smoke[0]['top_k']).style.format({'confidence': '{:.2%}'}))


## External Test: สร้างชุดที่ label ตรงกับ DenseNet แล้วประเมิน

ชุด `burapha_72` เดิมมีรหัส `175` แต่ DenseNet ต้องการ `174_ฎ` Notebook จะสร้างชุดใหม่จากไฟล์ต้นทาง Burapha/ALICE/iApp ที่อยู่ใน `data/unseen_test/burapha_source/` และใช้ label จาก Final checkpoint โดยตรง หากไม่มีต้นทางจะหยุดและแจ้งไฟล์ที่ขาด ชุดใหม่อยู่ใน `data/unseen_test/burapha_72_densenet/`

ผล External เป็นการประเมินชุดสาธารณะ 3,600 ภาพ มี exact-overlap check เทียบ training manifest ระหว่างสร้าง แต่ยังต้อง **ตรวจภาพ crop ของ `ๅ` ด้วยคน** และยังไม่ได้ยืนยัน near-duplicate หรือ validation overlap จึงไม่ควรเรียกว่า fully independent test


In [ ]:
if not CHECK_ONLY:
    import csv
    from collections import Counter
    from src.build_public_test import build_public_test, tis_code

    os.chdir(ROOT)  # build_public_test resolves its cached source archives from repository root
    EXTERNAL_DIR = ROOT / 'data/unseen_test/burapha_72_densenet'
    manifest_path = EXTERNAL_DIR / 'manifest.csv'
    if not manifest_path.exists():
        build_public_test(labels_path=FINAL_DIR / 'label_to_index.json', output_dir=EXTERNAL_DIR,
                          per_class=50, train_manifest=ROOT / 'data/splits/1BnPkvJJlE7QDZ0sgEDujS23Pr8Cj3IKq_source_v3/train.csv')
    with manifest_path.open(encoding='utf-8', newline='') as handle:
        external_rows = list(csv.DictReader(handle))
    code_to_label = {str(tis_code(label)): label for label in predictor.labels}
    counts = Counter(row['label'] for row in external_rows)
    if len(code_to_label) != 72 or set(counts) != set(code_to_label) or len(external_rows) != 3600 or set(counts.values()) != {50}:
        raise ValueError('External manifest is not 50 images × the exact 72 model labels')
    paths = [EXTERNAL_DIR / row['path'] for row in external_rows]
    if len(set(paths)) != len(paths) or any(not path.is_file() for path in paths):
        raise ValueError('External images have missing or duplicate paths')
    build_report = json.loads((EXTERNAL_DIR / 'report.json').read_text(encoding='utf-8'))
    if build_report['exact_duplicate_images'] or build_report['exact_training_overlaps']:
        raise ValueError('External set has exact duplicates or train overlaps')
    print('External dataset:', len(external_rows), 'images,', len(counts), 'classes')
    print('Review crop sheet:', EXTERNAL_DIR / 'review_55_ๅ.png')


In [ ]:
if not CHECK_ONLY:
    from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

    EXTERNAL_RESULTS = OUTPUT_DIR / 'external'
    EXTERNAL_RESULTS.mkdir(parents=True, exist_ok=True)
    saved_predictions = EXTERNAL_RESULTS / 'predictions.json'
    if saved_predictions.exists():
        payload = json.loads(saved_predictions.read_text(encoding='utf-8'))
        if payload['checkpoint_sha256'] != predictor.sha256 or payload['manifest_sha256'] != hashlib.sha256(manifest_path.read_bytes()).hexdigest():
            raise ValueError('Saved external results belong to different checkpoint or manifest')
        predictions = payload['rows']
    else:
        predictions = predictor.predict(paths, top_k=3, tta='none')
        saved_predictions.write_text(json.dumps({'checkpoint_sha256': predictor.sha256,
            'manifest_sha256': hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
            'rows': predictions}, ensure_ascii=False, indent=2), encoding='utf-8')
    if len(predictions) != len(external_rows) or any(row['status'] != 'OK' for row in predictions):
        raise RuntimeError('External inference incomplete or image unreadable')
    if [Path(row['path']) for row in predictions] != paths:
        raise ValueError('Saved predictions do not align with external manifest paths')
    y_true = [code_to_label[row['label']] for row in external_rows]
    y_pred = [row['prediction'] for row in predictions]
    labels = predictor.labels
    metrics = {'checkpoint_sha256': predictor.sha256, 'images': len(y_true),
               'classes': len(labels), 'status': 'preliminary_crop_review_pending',
               'dataset': 'burapha_72_densenet', 'exact_training_overlaps': build_report['exact_training_overlaps'],
               'near_duplicate_audit': 'not_done', 'validation_overlap_audit': 'not_done',
               'accuracy': accuracy_score(y_true, y_pred),
               'macro_precision': precision_score(y_true, y_pred, labels=labels, average='macro', zero_division=0),
               'macro_recall': recall_score(y_true, y_pred, labels=labels, average='macro', zero_division=0),
               'macro_f1': f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0),
               'balanced_accuracy': balanced_accuracy_score(y_true, y_pred)}
    metrics['validation_accuracy'] = selected['val_accuracy']
    metrics['validation_macro_f1'] = selected['val_macro_f1']
    metrics['generalization_gap_accuracy'] = metrics['validation_accuracy'] - metrics['accuracy']
    metrics['generalization_gap_macro_f1'] = metrics['validation_macro_f1'] - metrics['macro_f1']
    metrics = {key: float(value) if isinstance(value, float) else value for key, value in metrics.items()}
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    pd.DataFrame(matrix, index=labels, columns=labels).to_csv(EXTERNAL_RESULTS / 'confusion_matrix.csv')
    per_class = pd.DataFrame({'label': labels, 'recall': recall_score(y_true, y_pred, labels=labels, average=None, zero_division=0)})
    per_class.to_csv(EXTERNAL_RESULTS / 'per_class_recall.csv', index=False)
    pd.DataFrame({'path': [str(p) for p in paths], 'true_label': y_true, 'prediction': y_pred,
                  'confidence': [row['confidence'] for row in predictions]}).to_csv(EXTERNAL_RESULTS / 'predictions.csv', index=False)
    (EXTERNAL_RESULTS / 'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
    display(pd.DataFrame({'metric': [k for k in metrics if k not in {'checkpoint_sha256', 'images', 'classes'}],
                          'value': [v for k, v in metrics.items() if k not in {'checkpoint_sha256', 'images', 'classes'}]}))
    display(per_class.sort_values('recall').head(15))
    print('Saved report:', EXTERNAL_RESULTS / 'metrics.json')


## งานที่ต้องดูหลัง Run All

ผลใน Notebook ใช้เป็น **preliminary external result** จนกว่าจะเปิด `data/unseen_test/burapha_72_densenet/review_55_ๅ.png` ตรวจ crop ของ `ๅ` และบันทึกผลการตรวจ นอกจากนี้ยังไม่มี near-duplicate/validation overlap audit ของชุดสาธารณะ

หาก E5 เป็นข้อบังคับของรายงาน ต้องกลับไปดู `error_analysis/index.html` กำหนดวิธีแก้เฉพาะคลาสจากภาพผิด แล้วฝึก E5 ก่อนเลือก Final ใหม่ ผล External รอบนี้ห้ามใช้ย้อนกลับไปกำหนด E5

Notebook เขียนผลลงไฟล์ CSV/JSON และแสดงใน output ของเซลล์; Jupyter จะบันทึก output ลง `.ipynb` เมื่อกด Save/Save All
